### 01. Implementing modelling

First we want to create a simple linear regression to serve as our baseline.

In [2]:
# First we got to do some imports (I'm just grabbing the ones from 02 for now)
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression


# Random seed for reproducibility
np.random.seed(42)

df = pd.read_csv("data/ecowas_df_synthetic_full.csv")

In [3]:
df.columns

Index(['Unnamed: 0', 'country', 'year', 'disorder_Demonstrations',
       'disorder_Political violence',
       'disorder_Political violence; Demonstrations',
       'disorder_Strategic developments', 'event_Battles',
       'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
       'event_Strategic developments', 'event_Violence against civilians',
       'perpetrator_Civilians', 'perpetrator_External/Other forces',
       'perpetrator_Identity militia', 'perpetrator_Political militia',
       'perpetrator_Protesters', 'perpetrator_Rebel group',
       'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
       'target_External/Other forces', 'target_Identity militia',
       'target_Political militia', 'target_Protesters', 'target_Rebel group',
       'target_Rioters', 'target_State forces', 'fatalities', 'iso3_o',
       'iso3_d', 'distw_harmonic', 'distw_arithmetic', 'dist', 'distcap',
       'contig', 'diplo_disagreement', 'comlang_off', 'comlang

In [12]:
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

In [74]:
test = ["hello", "this", "is", "a", "test"]

test.remove("a")

#print(len(test)+1)

for i in range(0, len(test)+1, 2):
    print(i) 
    print(i+1)


0
1
2
3
4
5


### Log/Linear Regression on Economic/Gravity Data

This will serve as our primary baseline. If the machine learning model does not improve on this, then it does not add predictive power beyond simple Gravity/trade fundamentals

In [87]:
legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic"}

def baseline_linear(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016):
    '''
    The main function for creating our regression

    
    Input:
        df: pd.DataFrame -
            The main dataframe to work with.
        target_variable: str -
            The column which we are inferring for. 
        columns: list -
            A list of column elements that are included in the model. NOTE: Always include the basic Gravity columns, other we raise an error
        year_split: int -
            The year where we split between train and test. Defaults to 2016
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()
    
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of the following columns from dataframe: {required_columns}.")
    
    # We check for validity in input
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. \n"
                         f"Target variable must be one of {legal_target_vars}.")
    
    


    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df = test_df.dropna()


    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    

    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]


    model = LinearRegression()
    model.fit(X_train, y_train)


    # Finally we can get some workable metrics from this regression:
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Out-of-sample RMSE:", rmse)
    print("Out-of-sample R²:", r2)



    groups = train_df_full.loc[train_df_model.index, "dyad"]
    X = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                                cov_kwds={"groups": groups},)

    print(ols.summary())


    return target_variable
    


In [88]:
columns = [
    "year",
    "gdp_o", "gdp_d",
    "distw_arithmetic",
    "contig",
    "comlang_off",
]

baseline_linear(df, "tradeflow_baci", columns)

Out-of-sample RMSE: 2.4179838553739534
Out-of-sample R²: 0.3719398067722385
                               OLS Regression Results                               
Dep. Variable:     tradeflow_baci_log_trade   R-squared:                       0.384
Model:                                  OLS   Adj. R-squared:                  0.383
Method:                       Least Squares   F-statistic:                     58.30
Date:                      Thu, 16 Apr 2026   Prob (F-statistic):           6.82e-29
Time:                              22:15:00   Log-Likelihood:                -8307.6
No. Observations:                      3640   AIC:                         1.663e+04
Df Residuals:                          3633   BIC:                         1.667e+04
Df Model:                                 6                                         
Covariance Type:                    cluster                                         
                  coef    std err          z      P>|z|      [0.025      0

'tradeflow_baci'

In [ ]:
# First we try a simple Gravity Trade implementation, to show the validity of the data in our dataset.
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]


# We get the standards for this kind of modelling:
columns = [
    "year",
    "tradeflow_baci",
    "gdp_o", "gdp_d",
    "distw_arithmetic",
    "contig",
    "comlang_off",
    "dyad"
]


train_df = train_df[columns]
test_df = test_df[columns]

train_df.dropna()
test_df.dropna()


# As per usual, we want to look at log-trade. 
train_df["baci_log_trade"] = np.log(train_df["tradeflow_baci"])
test_df["baci_log_trade"]  = np.log(test_df["tradeflow_baci"])

train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

X_train = train_df[
    ["log_gdp_o", "log_gdp_d", "log_distw", "contig", "comlang_off", "year"]
]
y_train = train_df["baci_log_trade"]

X_test = test_df[
    ["log_gdp_o", "log_gdp_d", "log_distw", "contig", "comlang_off", "year"]
]
y_test = test_df["baci_log_trade"]

model = LinearRegression()
model.fit(X_train, y_train)


# Finally we can get some workable metrics from this regression:
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Out-of-sample RMSE:", rmse)
print("Out-of-sample R²:", r2)


X = sm.add_constant(X_train)
ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                             cov_kwds={"groups": train_df["dyad"]})

print(ols.summary())

Out-of-sample RMSE: 2.417983855373954
Out-of-sample R²: 0.37193980677223837
                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.384
Model:                            OLS   Adj. R-squared:                  0.383
Method:                 Least Squares   F-statistic:                     58.30
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           6.82e-29
Time:                        13:57:46   Log-Likelihood:                -8307.6
No. Observations:                3640   AIC:                         1.663e+04
Df Residuals:                    3633   BIC:                         1.667e+04
Df Model:                           6                                         
Covariance Type:              cluster                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------

### Linear Regression on Conflict Data

We will try to establish a baseline using the conflict data as well. This should be a combination of the trade model above along with a subset of conflict data (as too many regressors will break the model) 

In [ ]:
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

# We can get all the ACLED conflict columns
columns = [
    "year",
    'disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities',

    "distw_arithmetic", "tradeflow_baci", "contig", "comlang_off", "dyad"
]


train_df = train_df[columns]
test_df = test_df[columns]

train_df.dropna()
test_df.dropna()


# As per usual, we want to look at log-trade. 
train_df["baci_log_trade"] = np.log(train_df["tradeflow_baci"])
test_df["baci_log_trade"]  = np.log(test_df["tradeflow_baci"])


train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])
test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

X_train = train_df[
    ['disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities', "log_distw", "contig", "comlang_off", "year"]
]
y_train = train_df["baci_log_trade"]

X_test = test_df[
    ['disorder_Demonstrations',
    'disorder_Political violence',
    'disorder_Political violence; Demonstrations',
    'disorder_Strategic developments', 'event_Battles',
    'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
    'event_Strategic developments', 'event_Violence against civilians',
    'perpetrator_Civilians', 'perpetrator_External/Other forces',
    'perpetrator_Identity militia', 'perpetrator_Political militia',
    'perpetrator_Protesters', 'perpetrator_Rebel group',
    'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
    'target_External/Other forces', 'target_Identity militia',
    'target_Political militia', 'target_Protesters', 'target_Rebel group',
    'target_Rioters', 'target_State forces', 'fatalities', "log_distw", "contig", "comlang_off", "year"]
]
y_test = test_df["baci_log_trade"]

model = LinearRegression()
model.fit(X_train, y_train)


# Finally we can get some workable metrics from this regression:
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Out-of-sample RMSE:", rmse)
print("Out-of-sample R²:", r2)


X = sm.add_constant(X_train)
ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                             cov_kwds={"groups": train_df["dyad"]})

print(ols.summary())

Out-of-sample RMSE: 7.0189515505173885
Out-of-sample R²: -4.29223309286898
                            OLS Regression Results                            
Dep. Variable:         baci_log_trade   R-squared:                       0.227
Model:                            OLS   Adj. R-squared:                  0.221
Method:                 Least Squares   F-statistic:                     14.00
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           3.71e-22
Time:                        14:02:42   Log-Likelihood:                -8720.7
No. Observations:                3640   AIC:                         1.750e+04
Df Residuals:                    3611   BIC:                         1.768e+04
Df Model:                          28                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------

c:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\bachelor_2026\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 31, but rank is 28
  warnings.warn('covariance of constraints does not have full '
